# ✂️ 토크나이저(Tokenizer): 처음 개념부터 코드 실습까지

목표는 **완전 초보 개념 이해 → ChatGPT로 질문하며 확인 → 직접 코드로 토크나이저 만들기**까지 자연스럽게 올라가는 것입니다.

---

## ChatGPT는 문장을 어떻게 읽기 시작할까?

ChatGPT 같은 AI는 문장을 통째로 바로 이해하지 않습니다. 먼저 텍스트를 작은 조각인 **토큰(token)** 으로 나누고, 각 토큰을 숫자 ID로 바꿉니다.

```text
입력: "나는 AI를 배운다"
토큰: ["나는", "AI", "를", "배운다"]
ID:   [12, 305, 8, 91]
```

이 노트북에서는 다음 순서로 배웁니다.
- 토크나이저가 왜 필요한지 이해하기
- 단어, 문자, 서브워드 방식 비교하기
- 한국어 토큰화가 왜 어려운지 관찰하기
- 토큰을 숫자 ID로 바꾸고 다시 복원하기
- 작은 `MiniTokenizer` 클래스로 코드 레벨까지 가기

**핵심 메시지**: *토크나이저는 AI가 글을 계산 가능한 조각으로 바꾸는 첫 번째 입구입니다.*

## 1단계: Beginner 개념 — 토크나이저가 뭐예요?

토크나이저는 큰 문장을 작은 조각으로 나누는 도구입니다.

비유하면 요리 재료를 손질하는 과정과 비슷합니다.
- 긴 문장 = 손질 전 재료
- 토큰 = 먹기 좋게 자른 조각
- 토크나이저 = 재료를 자르는 방법

먼저 ChatGPT에게 아래 질문을 던져 개념을 확인해보세요.

```text
고등학생에게 설명하듯이 토크나이저(tokenizer)가 무엇인지 쉽게 알려주세요.
일상생활 비유를 하나 쓰고, "토큰"이라는 말도 함께 설명해주세요.
```

그리고 바로 코드로 같은 문장이 어떻게 나뉘는지 봅니다.

In [1]:
texts = [
    "안녕하세요 여러분!",
    "I am learning AI and machine learning.",
    "파이썬 programming은 정말 재미있어요 😊",
    "오늘 시험이 너무 어려웠다... 그래도 다시 공부하자!",
    "ChatGPT는 text를 token으로 나눕니다.",
]

print(f"📝 예시 문장 수: {len(texts)}개")
for i, text in enumerate(texts, 1):
    print(f"{i}. {text}")


📝 예시 문장 수: 5개
1. 안녕하세요 여러분!
2. I am learning AI and machine learning.
3. 파이썬 programming은 정말 재미있어요 😊
4. 오늘 시험이 너무 어려웠다... 그래도 다시 공부하자!
5. ChatGPT는 text를 token으로 나눕니다.


## 2단계: 왜 필요할까? — AI는 문자열보다 숫자를 다룬다

AI 모델은 글자를 그대로 계산하지 않습니다. 보통 이런 흐름을 거칩니다.

1. 문장을 토큰으로 나눈다
2. 토큰을 숫자 ID로 바꾼다
3. 숫자 ID를 임베딩 벡터로 바꾼다
4. 모델이 벡터를 계산해 다음 토큰을 예측한다

ChatGPT에게 확인 질문:

```text
ChatGPT가 내 질문을 받았을 때, 토크나이저가 어떤 역할을 하는지 단계별로 설명해주세요.
텍스트가 토큰이 되고 숫자가 되는 과정을 고등학생 눈높이로 알려주세요.
```

이제 가장 단순한 방식부터 직접 구현합니다.

In [2]:
def whitespace_tokenize(text):
    return text.split()

print("🔹 공백 기반 토큰화")
for text in texts:
    tokens = whitespace_tokenize(text)
    print(f"\n입력: {text}")
    print(f"토큰({len(tokens)}개): {tokens}")


🔹 공백 기반 토큰화

입력: 안녕하세요 여러분!
토큰(2개): ['안녕하세요', '여러분!']

입력: I am learning AI and machine learning.
토큰(7개): ['I', 'am', 'learning', 'AI', 'and', 'machine', 'learning.']

입력: 파이썬 programming은 정말 재미있어요 😊
토큰(5개): ['파이썬', 'programming은', '정말', '재미있어요', '😊']

입력: 오늘 시험이 너무 어려웠다... 그래도 다시 공부하자!
토큰(7개): ['오늘', '시험이', '너무', '어려웠다...', '그래도', '다시', '공부하자!']

입력: ChatGPT는 text를 token으로 나눕니다.
토큰(4개): ['ChatGPT는', 'text를', 'token으로', '나눕니다.']


## 3단계: 공백 기반의 한계 — 구두점과 한국어

공백 기준 토큰화는 쉽지만 한계가 있습니다.

예를 들어 `여러분!`은 단어와 느낌표가 붙어 있고, `programming은`은 영어 단어와 한국어 조사가 붙어 있습니다.

정규표현식을 사용하면 단어와 구두점을 조금 더 잘 분리할 수 있습니다.

In [3]:
import re


def regex_tokenize(text):
    # 영어 묶음, 숫자 묶음, 한글 묶음, 공백이 아닌 기호를 각각 토큰으로 잡습니다.
    return re.findall(r"[A-Za-z]+|[0-9]+|[가-힣]+|[^\w\s]", text)

print("🔍 정규표현식 기반 토큰화")
for text in texts:
    tokens = regex_tokenize(text)
    print(f"\n입력: {text}")
    print(f"토큰({len(tokens)}개): {tokens}")


🔍 정규표현식 기반 토큰화

입력: 안녕하세요 여러분!
토큰(3개): ['안녕하세요', '여러분', '!']

입력: I am learning AI and machine learning.
토큰(8개): ['I', 'am', 'learning', 'AI', 'and', 'machine', 'learning', '.']

입력: 파이썬 programming은 정말 재미있어요 😊
토큰(6개): ['파이썬', 'programming', '은', '정말', '재미있어요', '😊']

입력: 오늘 시험이 너무 어려웠다... 그래도 다시 공부하자!
토큰(11개): ['오늘', '시험이', '너무', '어려웠다', '.', '.', '.', '그래도', '다시', '공부하자', '!']

입력: ChatGPT는 text를 token으로 나눕니다.
토큰(8개): ['ChatGPT', '는', 'text', '를', 'token', '으로', '나눕니다', '.']


## 4단계: 토큰화 방식 1 — 단어 기반

단어 기반 토크나이저는 문장을 단어 단위로 나눕니다.

장점:
- 사람이 이해하기 쉽다
- 토큰 수가 비교적 적다

단점:
- 처음 보는 단어를 처리하기 어렵다
- 한국어처럼 조사와 어미가 붙는 언어에서는 애매해진다

ChatGPT 질문:

```text
단어 기반 토크나이저의 장점과 단점을 고등학생도 이해할 수 있게 예시와 함께 설명해주세요.
특히 처음 보는 단어(OOV)가 왜 문제가 되는지도 알려주세요.
```

In [4]:
def word_tokenize(text):
    return regex_tokenize(text)

word_examples = [
    "Hello, world! How are you?",
    "학교에서 친구들과 공부했어요.",
    "ChatGPT tokenization is interesting!",
]

print("📚 단어/기호 기반 토큰화")
for text in word_examples:
    tokens = word_tokenize(text)
    print(f"\n입력: {text}")
    print(f"토큰({len(tokens)}개): {tokens}")


📚 단어/기호 기반 토큰화

입력: Hello, world! How are you?
토큰(8개): ['Hello', ',', 'world', '!', 'How', 'are', 'you', '?']

입력: 학교에서 친구들과 공부했어요.
토큰(4개): ['학교에서', '친구들과', '공부했어요', '.']

입력: ChatGPT tokenization is interesting!
토큰(5개): ['ChatGPT', 'tokenization', 'is', 'interesting', '!']


## 5단계: 토큰화 방식 2 — 문자 기반

문자 기반 토크나이저는 한 글자씩 나눕니다.

장점:
- 어떤 단어든 처리할 수 있다
- 구현이 매우 쉽다

단점:
- 토큰 수가 너무 많아진다
- 단어의 의미가 너무 잘게 쪼개진다

문자 기반은 “모르는 단어 문제”는 줄이지만, 긴 글을 처리할 때 비효율적일 수 있습니다.

In [5]:
def char_tokenize(text, keep_space=False):
    if keep_space:
        return list(text)
    return [ch for ch in text if not ch.isspace()]

sample = "ChatGPT는 토큰을 나눕니다!"
print("🔤 문자 기반 토큰화")
print("입력:", sample)
print("공백 제외:", char_tokenize(sample))
print("공백 포함:", char_tokenize(sample, keep_space=True))
print("공백 제외 토큰 수:", len(char_tokenize(sample)))
print("공백 포함 토큰 수:", len(char_tokenize(sample, keep_space=True)))


🔤 문자 기반 토큰화
입력: ChatGPT는 토큰을 나눕니다!
공백 제외: ['C', 'h', 'a', 't', 'G', 'P', 'T', '는', '토', '큰', '을', '나', '눕', '니', '다', '!']
공백 포함: ['C', 'h', 'a', 't', 'G', 'P', 'T', '는', ' ', '토', '큰', '을', ' ', '나', '눕', '니', '다', '!']
공백 제외 토큰 수: 16
공백 포함 토큰 수: 18


## 6단계: 토큰화 방식 3 — 서브워드

현대 LLM은 보통 단어와 문자 사이에 있는 **서브워드(subword)** 방식을 많이 씁니다.

예를 들어 영어에서는 이런 식으로 생각할 수 있습니다.

```text
unhappiness → un + happi + ness
learning    → learn + ing
```

실제 BPE, WordPiece, SentencePiece는 데이터에서 자주 등장하는 조각을 학습합니다. 여기서는 원리를 보기 위해 단순한 규칙으로 흉내냅니다.

ChatGPT 질문:

```text
서브워드 토크나이저가 단어 기반과 문자 기반의 중간 방식이라고 들었습니다.
왜 현대 AI 모델에서 많이 쓰이는지 쉬운 예시로 설명해주세요.
```

In [6]:
prefixes = ["un", "re", "pre", "anti"]
suffixes = ["ing", "ed", "er", "est", "ness", "ly", "s"]


def simple_subword_tokenize_word(word):
    lower = word.lower()
    pieces = []

    for prefix in prefixes:
        if lower.startswith(prefix) and len(lower) > len(prefix) + 2:
            pieces.append(word[:len(prefix)])
            word = word[len(prefix):]
            lower = lower[len(prefix):]
            break

    suffix_piece = None
    for suffix in suffixes:
        if lower.endswith(suffix) and len(lower) > len(suffix) + 2:
            suffix_piece = word[-len(suffix):]
            word = word[:-len(suffix)]
            break

    if word:
        pieces.append(word)
    if suffix_piece:
        pieces.append("##" + suffix_piece)
    return pieces

english_words = ["learning", "unhappy", "teacher", "replaying", "kindness", "students"]

print("⚡ 단순 서브워드 토큰화")
for word in english_words:
    print(f"{word:>10} → {simple_subword_tokenize_word(word)}")


⚡ 단순 서브워드 토큰화
  learning → ['learn', '##ing']
   unhappy → ['un', 'happy']
   teacher → ['teach', '##er']
 replaying → ['re', 'play', '##ing']
  kindness → ['kind', '##ness']
  students → ['student', '##s']


## 7단계: 같은 문장, 다른 토크나이저 비교

이제 같은 문장을 여러 방식으로 나눠봅니다.

관찰할 점:
- 공백 기반은 빠르지만 단순함
- 정규표현식 기반은 구두점과 단어를 분리함
- 문자 기반은 가장 잘게 나눔
- 서브워드는 단어 내부까지 적당히 나눔

토큰화 방식이 달라지면 토큰 수, 처리 비용, 모델이 보는 정보의 단위도 달라집니다.

In [7]:
def simple_subword_tokenize(text):
    output = []
    for token in regex_tokenize(text):
        if re.fullmatch(r"[A-Za-z]+", token):
            output.extend(simple_subword_tokenize_word(token))
        else:
            output.append(token)
    return output

compare_text = "I am learning ChatGPT tokenization!"
methods = {
    "공백": whitespace_tokenize,
    "정규표현식": regex_tokenize,
    "문자": char_tokenize,
    "서브워드": simple_subword_tokenize,
}

print("⚖️ 토큰화 방식 비교")
print("입력:", compare_text)
for name, fn in methods.items():
    tokens = fn(compare_text)
    print(f"\n[{name}] {len(tokens)}개")
    print(tokens)


⚖️ 토큰화 방식 비교
입력: I am learning ChatGPT tokenization!

[공백] 5개
['I', 'am', 'learning', 'ChatGPT', 'tokenization!']

[정규표현식] 6개
['I', 'am', 'learning', 'ChatGPT', 'tokenization', '!']

[문자] 31개
['I', 'a', 'm', 'l', 'e', 'a', 'r', 'n', 'i', 'n', 'g', 'C', 'h', 'a', 't', 'G', 'P', 'T', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n', '!']

[서브워드] 7개
['I', 'am', 'learn', '##ing', 'ChatGPT', 'tokenization', '!']


## 8단계: 한국어 토큰화 — 조사와 어미 때문에 더 복잡하다

한국어는 영어보다 토큰화가 까다로운 경우가 많습니다.

예를 들어 `학교에서`는 `학교 + 에서`로 볼 수 있고, `공부했어요`는 `공부 + 했어요`처럼 볼 수도 있습니다.

여기서는 전문 형태소 분석기 대신, 자주 쓰는 조사만 간단히 떼어보는 작은 규칙 기반 토크나이저를 만들어봅니다.

ChatGPT 질문:

```text
한국어 토큰화가 영어보다 어려운 이유를 조사와 어미 예시로 설명해주세요.
"학교에서 공부했어요"를 예로 들어주세요.
```

In [8]:
particles = ["에서", "에게", "으로", "부터", "까지", "를", "을", "이", "가", "은", "는", "도", "만", "와", "과", "에"]


def simple_korean_tokenize(text):
    raw_tokens = regex_tokenize(text)
    result = []
    for token in raw_tokens:
        if not re.fullmatch(r"[가-힣]+", token):
            result.append(token)
            continue
        matched = False
        for particle in sorted(particles, key=len, reverse=True):
            if token.endswith(particle) and len(token) > len(particle):
                result.append(token[:-len(particle)])
                result.append(particle)
                matched = True
                break
        if not matched:
            result.append(token)
    return result

korean_examples = [
    "학교에서 공부했어요",
    "친구와 운동장에서 축구를 했다",
    "선생님에게 질문을 했다",
    "ChatGPT는 한국어와 English를 함께 처리한다",
]

print("🇰🇷 한국어 간단 토큰화 비교")
for text in korean_examples:
    print(f"\n입력: {text}")
    print("정규표현식:", regex_tokenize(text))
    print("조사 분리:", simple_korean_tokenize(text))


🇰🇷 한국어 간단 토큰화 비교

입력: 학교에서 공부했어요
정규표현식: ['학교에서', '공부했어요']
조사 분리: ['학교', '에서', '공부했어요']

입력: 친구와 운동장에서 축구를 했다
정규표현식: ['친구와', '운동장에서', '축구를', '했다']
조사 분리: ['친구', '와', '운동장', '에서', '축구', '를', '했다']

입력: 선생님에게 질문을 했다
정규표현식: ['선생님에게', '질문을', '했다']
조사 분리: ['선생님', '에게', '질문', '을', '했다']

입력: ChatGPT는 한국어와 English를 함께 처리한다
정규표현식: ['ChatGPT', '는', '한국어와', 'English', '를', '함께', '처리한다']
조사 분리: ['ChatGPT', '는', '한국어', '와', 'English', '를', '함께', '처리한다']


## 9단계: 어휘집과 토큰 ID — 토큰을 숫자로 바꾸기

AI 모델은 문자열 자체보다 숫자 ID를 다룹니다.

토크나이저는 보통 다음 일을 합니다.

1. 텍스트를 토큰으로 나눈다
2. 각 토큰을 어휘집에서 찾는다
3. 토큰을 숫자 ID로 바꾼다
4. 필요하면 ID를 다시 토큰으로 복원한다

어휘집에 없는 토큰은 `[UNK]` 같은 특별 토큰으로 처리할 수 있습니다.

In [9]:
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

all_tokens = []
for text in texts + korean_examples:
    all_tokens.extend(simple_korean_tokenize(text))

vocab_tokens = special_tokens + sorted(set(all_tokens))
token_to_id = {token: i for i, token in enumerate(vocab_tokens)}
id_to_token = {i: token for token, i in token_to_id.items()}


def encode(text):
    tokens = ["[BOS]"] + simple_korean_tokenize(text) + ["[EOS]"]
    ids = [token_to_id.get(token, token_to_id["[UNK]"]) for token in tokens]
    return tokens, ids


def decode(ids):
    return [id_to_token.get(i, "[UNK]") for i in ids]

sample = "오늘 ChatGPT는 토큰을 나눕니다!"
tokens, ids = encode(sample)

print(f"📦 어휘집 크기: {len(vocab_tokens)}")
print("어휘 일부:", vocab_tokens[:18])
print("\n입력:", sample)
print("토큰:", tokens)
print("ID:", ids)
print("복원:", decode(ids))


📦 어휘집 크기: 52
어휘 일부: ['[PAD]', '[UNK]', '[BOS]', '[EOS]', '!', '.', 'AI', 'ChatGPT', 'English', 'I', 'am', 'and', 'learning', 'machine', 'programming', 'text', 'token', '공부하자']

입력: 오늘 ChatGPT는 토큰을 나눕니다!
토큰: ['[BOS]', '오늘', 'ChatGPT', '는', '토큰', '을', '나눕니다', '!', '[EOS]']
ID: [2, 33, 7, 22, 1, 38, 20, 4, 3]
복원: ['[BOS]', '오늘', 'ChatGPT', '는', '[UNK]', '을', '나눕니다', '!', '[EOS]']


## 10단계: 실제 도구 — OpenAI Tokenizer로 확인하기

[OpenAI Tokenizer](https://platform.openai.com/tokenizer)는 텍스트가 OpenAI 모델에서 어떻게 토큰으로 나뉘는지 직접 확인할 수 있는 공식 도구입니다.

이 도구에서는 다음을 볼 수 있습니다.

- 입력한 문장의 토큰 개수
- 전체 글자 수
- 텍스트가 어떤 조각으로 나뉘는지
- 모델 계열에 따라 토큰화가 어떻게 달라질 수 있는지

OpenAI 도움말에 따르면, 토큰은 모델이 처리하는 텍스트의 기본 단위입니다. 토큰은 한 글자처럼 짧을 수도 있고, 단어 하나처럼 길 수도 있으며, 공백, 구두점, 단어 일부도 토큰 수에 영향을 줍니다.

### 영어와 한국어 비교 실험

OpenAI Tokenizer에 아래 문장을 각각 넣어보고 토큰 수를 비교해보세요.

```text
I go to school every morning.
```

```text
나는 매일 아침 학교에 갑니다.
```

```text
ChatGPT로 한국어 token을 확인해요!
```

관찰할 점:

1. 영어는 보통 공백 기준 단어와 서브워드가 비교적 예측하기 쉽게 나뉩니다.
2. 한국어는 조사, 어미, 한글 음절 때문에 영어와 다른 방식으로 잘릴 수 있습니다.
3. 같은 의미의 문장이라도 언어에 따라 토큰 수가 달라질 수 있습니다.
4. 토큰 수는 API 비용, 입력 제한, 출력 제한과 연결됩니다.

OpenAI의 설명에 따르면 영어에서는 대략 `1 token ≈ 4 characters`, `100 tokens ≈ 75 words` 정도로 볼 수 있습니다. 하지만 영어가 아닌 텍스트는 문자 대비 토큰 비율이 더 높아질 수 있으므로, 한국어 문장을 사용할 때도 실제 토큰 수를 확인하는 습관이 중요합니다.

### 이 노트북과 연결하기

우리가 만든 토크나이저는 아주 단순한 교육용 버전입니다.

반면 OpenAI Tokenizer는 실제 모델에서 사용하는 방식에 더 가까운 결과를 보여줍니다. 따라서 이 도구를 사용하면 다음을 확인할 수 있습니다.

- 우리가 직접 만든 토크나이저와 실제 토크나이저의 차이
- 한국어와 영어의 토큰 수 차이
- 공백, 구두점, 대소문자가 토큰화에 미치는 영향
- 토큰 수가 왜 모델 사용 비용과 문맥 길이에 영향을 주는지

### 직접 해볼 활동

1. [OpenAI Tokenizer](https://platform.openai.com/tokenizer)에 접속합니다.
2. 영어 문장과 한국어 문장을 하나씩 입력합니다.
3. 토큰 수와 글자 수를 기록합니다.
4. 같은 의미의 영어/한국어 문장을 비교합니다.
5. 구두점, 이모지, 띄어쓰기를 바꿔보며 토큰 수가 어떻게 달라지는지 관찰합니다.

### 💬 ChatGPT에게 확인질문
```text
OpenAI Tokenizer에서 영어 문장과 한국어 문장의 토큰 수가 다르게 나오는 이유를 설명해주세요.
토큰이 글자, 단어, 단어 일부, 공백, 구두점을 포함할 수 있다는 점과 API 비용/문맥 길이에 미치는 영향도 함께 알려주세요.
```

참고: [OpenAI Tokenizer](https://platform.openai.com/tokenizer), [OpenAI Help: What are tokens and how to count them?](https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them)


## 11단계: 토큰 수가 비용과 문맥 길이에 미치는 영향

LLM에서는 토큰 수가 중요합니다.

토큰 수가 많아지면:
- 처리 시간이 늘어날 수 있음
- 비용이 늘어날 수 있음
- 모델이 한 번에 볼 수 있는 문맥 길이를 더 빨리 사용함

그래서 좋은 토크나이저는 의미를 잘 보존하면서도 너무 많은 토큰을 만들지 않아야 합니다.

In [10]:
long_text = "오늘은 AI 수업에서 토크나이저와 임베딩을 배웠고, ChatGPT가 문장을 어떻게 처리하는지 실험했다."

count_table = []
for name, fn in [
    ("공백", whitespace_tokenize),
    ("정규표현식", regex_tokenize),
    ("문자", char_tokenize),
    ("한국어 조사 분리", simple_korean_tokenize),
]:
    tokens = fn(long_text)
    count_table.append((name, len(tokens), tokens[:14]))

print("📊 토큰 수 비교")
for name, count, preview in count_table:
    print(f"{name:>10}: {count:>2}개 | 미리보기: {preview}")


📊 토큰 수 비교
        공백: 11개 | 미리보기: ['오늘은', 'AI', '수업에서', '토크나이저와', '임베딩을', '배웠고,', 'ChatGPT가', '문장을', '어떻게', '처리하는지', '실험했다.']
     정규표현식: 14개 | 미리보기: ['오늘은', 'AI', '수업에서', '토크나이저와', '임베딩을', '배웠고', ',', 'ChatGPT', '가', '문장을', '어떻게', '처리하는지', '실험했다', '.']
        문자: 47개 | 미리보기: ['오', '늘', '은', 'A', 'I', '수', '업', '에', '서', '토', '크', '나', '이', '저']
 한국어 조사 분리: 19개 | 미리보기: ['오늘', '은', 'AI', '수업', '에서', '토크나이저', '와', '임베딩', '을', '배웠고', ',', 'ChatGPT', '가', '문장']


## 12단계: 코드 레벨 — 지금까지 배운 내용을 MiniTokenizer로 정리하기

이제 지금까지 만든 기능을 작은 클래스로 묶어봅니다.

앞 단계에서 우리는 실제 OpenAI Tokenizer로 토큰 수를 관찰했고, 토큰 수가 비용과 문맥 길이에 영향을 준다는 것도 확인했습니다. 이제 그 흐름을 코드 관점에서 정리해봅니다.

이 클래스는 실제 토크나이저에 비하면 매우 단순하지만, 핵심 구조는 비슷합니다.
- `tokenize`: 텍스트를 토큰으로 나눔
- `fit`: 어휘집 만들기
- `encode`: 토큰을 ID로 바꿈
- `decode`: ID를 다시 토큰으로 복원
- `analyze`: 토큰 수와 ID를 한 번에 확인

이 단계가 beginner에서 code level로 넘어가는 최종 정리입니다.

In [11]:
class MiniTokenizer:
    def __init__(self, tokenize_fn, special_tokens=None):
        self.tokenize_fn = tokenize_fn
        self.special_tokens = special_tokens or ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
        self.token_to_id = {}
        self.id_to_token = {}

    def fit(self, texts):
        tokens = []
        for text in texts:
            tokens.extend(self.tokenize_fn(text))
        vocab = self.special_tokens + sorted(set(tokens))
        self.token_to_id = {token: i for i, token in enumerate(vocab)}
        self.id_to_token = {i: token for token, i in self.token_to_id.items()}

    def tokenize(self, text):
        return self.tokenize_fn(text)

    def encode(self, text, add_special_tokens=True):
        tokens = self.tokenize(text)
        if add_special_tokens:
            tokens = ["[BOS]"] + tokens + ["[EOS]"]
        ids = [self.token_to_id.get(token, self.token_to_id["[UNK]"]) for token in tokens]
        return ids

    def decode(self, ids):
        return [self.id_to_token.get(i, "[UNK]") for i in ids]

    def analyze(self, text):
        tokens = self.tokenize(text)
        ids = self.encode(text)
        return {
            "text": text,
            "tokens": tokens,
            "token_count": len(tokens),
            "ids": ids,
            "decoded": self.decode(ids),
        }

mini = MiniTokenizer(simple_korean_tokenize)
mini.fit(texts + korean_examples)

analysis = mini.analyze("학교에서 ChatGPT를 공부했어요!")
print("🧪 MiniTokenizer 분석")
for key, value in analysis.items():
    print(f"{key}: {value}")


🧪 MiniTokenizer 분석
text: 학교에서 ChatGPT를 공부했어요!
tokens: ['학교', '에서', 'ChatGPT', '를', '공부했어요', '!']
token_count: 6
ids: [2, 47, 31, 7, 25, 18, 4, 3]
decoded: ['[BOS]', '학교', '에서', 'ChatGPT', '를', '공부했어요', '!', '[EOS]']


## 13단계: 실습 과제 — 토크나이저를 이해하기 위한 실험

아래 과제는 “토크나이저가 왜 필요한가”를 직접 느끼기 위한 실습입니다.

### 과제 1. 같은 문장을 여러 방식으로 나누기
아래 문장을 공백, 정규표현식, 문자, 한국어 조사 분리 방식으로 나눠보세요.

```text
ㅋㅋㅋ 대박! ChatGPT로 한국어 공부했어요.
```

질문:
```text
어떤 방식이 가장 보기 좋았나요? 어떤 방식이 토큰 수가 가장 많았나요?
```

### 과제 2. 한국어 조사 추가하기
`particles` 목록에 `랑`, `하고`, `처럼`을 추가해보세요. 결과가 어떻게 바뀌는지 관찰하세요.

### 과제 3. 어휘집에 없는 단어 실험하기
`mini.encode("새로운단어를 입력했어요")`를 실행해보세요. `[UNK]`가 어디에 나오는지 확인하세요.

### 과제 4. 나만의 MiniTokenizer 개선하기
아래 중 하나를 골라 개선해보세요.
- 이모지를 별도 토큰으로 더 잘 처리하기
- 영어는 소문자로 바꿔 저장하기
- 숫자를 `[NUMBER]` 토큰으로 바꾸기
- 조사뿐 아니라 간단한 어미도 분리하기

마지막에는 아래 문장을 완성해보세요.

```text
토크나이저가 달라지면 AI가 보는 입력도 달라진다. 왜냐하면 ____________.
```

## 📝 정리: Beginner에서 Code Level까지

| 단계 | 배운 내용 | 코드에서 한 일 |
|---|---|---|
| 개념 | 토크나이저는 문장을 작은 조각으로 나눔 | 예시 문장 준비 |
| 필요성 | AI는 텍스트를 토큰과 숫자로 처리 | 공백 토큰화 구현 |
| 단어/기호 | 구두점과 단어를 나눔 | `regex_tokenize` 구현 |
| 문자 기반 | 모든 글자를 처리할 수 있지만 길어짐 | `char_tokenize` 구현 |
| 서브워드 | 단어와 문자 사이의 절충 | 단순 서브워드 규칙 구현 |
| 한국어 | 조사와 어미 때문에 더 복잡함 | `simple_korean_tokenize` 구현 |
| 숫자화 | 토큰을 ID로 바꿈 | `encode`, `decode` 구현 |
| 실제 도구 | OpenAI Tokenizer로 실제 토큰화 관찰 | 영어/한국어 토큰 수 비교 활동 |
| 비용 감각 | 토큰 수가 문맥 길이와 비용에 영향 | 방식별 토큰 수 비교 |
| 코드 구조화 | 재사용 가능한 토크나이저 | `MiniTokenizer` 클래스 구현 |

전체 흐름은 이렇게 기억하면 됩니다.

1. 텍스트 입력 → 2. 토큰화 → 3. 토큰 ID 변환 → 4. 토큰 수 확인 → 5. 임베딩 → 6. 모델 계산 → 7. 다음 토큰 예측

> 토크나이저는 AI가 글을 읽기 전에 텍스트를 계산 가능한 단위로 정리해주는 입구입니다.